# FleetIQ / SmartTaxi — 06. Baseline Heuristic Modeling

## Objective
Fit and evaluate the Historical Average Speed heuristic baseline model (`src.models.baseline`).
Measures global urban speed and hourly segmented speeds to establish the lower-bound benchmark for ETA prediction.


In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.models.baseline import HistoricalAvgSpeedBaseline
from src.evaluation.metrics import evaluate_all

train_df = pd.read_parquet('../data/processed/train.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

baseline = HistoricalAvgSpeedBaseline(default_speed_kmh=25.0)
baseline.fit(
    distance_km=train_df['distance_haversine_km'],
    duration_min=train_df['trip_duration_minutes'],
    hours=train_df['pickup_hour'],
)

print(f'Fitted Global Speed: {baseline.global_speed_kmh:.2f} km/h')
print('Sample Hourly Speeds (km/h):', {h: round(baseline.hourly_speeds[h], 1) for h in range(0, 24, 4)})


Fitted Global Speed: 13.06 km/h
Sample Hourly Speeds (km/h): {0: 16.4, 4: 20.9, 8: 13.0, 12: 11.7, 16: 11.8, 20: 13.8}


## 1. Evaluate Baseline on Held-out Out-of-Time Test Set
We evaluate the fitted hourly baseline against actual trip durations on the October 2025 test split.


In [2]:
y_test = test_df['trip_duration_minutes'].values
preds = baseline.predict(test_df['distance_haversine_km'], hours=test_df['pickup_hour'])

metrics = evaluate_all(y_test, preds)
df_metrics = pd.DataFrame([metrics], index=['Baseline (Historical Avg Speed)'])
df_metrics[['mae_minutes', 'mae_seconds', 'rmse_minutes', 'r2', 'mape_pct']]


,mae_minutes,mae_seconds,rmse_minutes,r2,mape_pct
Baseline (Historical Avg Speed),7.907,474.4,13.136,0.2584,41.95


## 2. Conclusion Métier & Décision d'Architecture pour le Déploiement

### Constats Clés :
- **Faible pouvoir explicatif** : Le coefficient $R^2 = 0.258$ montre qu'une simple vitesse horaire n'explique qu'un quart de la variance des durées de courses.
- **Erreur élevée sur trajets longs** : L'erreur moyenne absolue (MAE) de **7.91 minutes** (474 secondes) est trop imprécise pour le dispatching en temps réel de SmartTaxi.

### Décision appliquée au Déploiement :
1. **Nécessité de modèles ML supervisés non-linéaires** : Cela justifie pleinement l'entraînement de modèles avancés (Random Forest, GBDT) dans le Notebook 07.
2. **Conservation en mode Secours (Fallback Heuristic)** : Cette baseline ne sera pas abandonnée ! Elle est conservée dans l'architecture de production (`src/models/baseline.py`).
3. Dans l'API FastAPI (`app.py`), si le modèle de Machine Learning rencontre un incident ou est indisponible, le microservice bascule instantanément sur cette heuristique urbaine avec le statut `FALLBACK_ROUTING` pour garantir la haute disponibilité sans jamais renvoyer d'erreur 500.
